# From Draft to Living Cell

_Investigation `draft-to-living-cell` — coder reproduction notebook._

**Question.** How far can a cell be built compositionally — assembled from typed interfaces,
each specified as a *draft* before any mechanism is chosen, then compiled into
something that actually runs — and does that composition hold together into a
whole cell that lives and dies?

An executable atlas of *A meta-modeler's guide to the cellular interface* (Agmon).
Each compositional figure (4–10) is rebuilt twice: as a semantic DRAFT composite —
typed interface ports + a behavior contract, no dynamics, rendered in the paper's
visual language beside the original illustration — and as a compiled EXECUTABLE
that runs. The two layers and the compiler between them (an algebraic effect
system) are the subject; the terminus is a composed whole cell.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-meta-modelers-guide/viva-meta-modelers-guide').is_dir():
    REPO = Path('/home/runner/work/viva-meta-modelers-guide/viva-meta-modelers-guide')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_meta_modelers_guide.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: The Typed Interface (`typed-interface`)

**Question.** Can the cellular boundary be specified as nothing but a set of typed, unit-bearing exchange ports (chemical, mechanical, electrical, thermal, plus growth rate, shape, objective, viability) with no committed mechanism, and then compiled — by installing one conforming handler — into a running, bounded, goal-directed cell whose interface is exactly the one declared?

**Objective.** Author the Fig 4b cellular interface as a draft composite of typed ports, install a bounded-cell handler via the compiler, run it, and measure whether the interface variables evolve as a goal-directed cell would.

**Hypothesis.** The Fig 4 interface authored as an inert draft (typed ports + a behavior contract, an update that does nothing) compiles under a single conforming handler into an executable that leaves every port and wire unchanged (law 2) yet produces genuine dynamics: shape and objective rise while the cell takes up chemical from its surroundings.

**Claim.** A cell's boundary is a small set of typed, unit-bearing ports; a conforming handler turns that inert interface into a bounded, goal-directed cell that grows (shape 1.0→4.2) and pursues an objective.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `interaction-modalities` | `viva_meta_modelers_guide.composites.fig04a-interaction-modalities` | 0 | — |
| `cellular-interface` | `viva_meta_modelers_guide.composites.fig04b-cellular-interface` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_meta_modelers_guide.composites.fig04a-interaction-modalities`** — `spec_viva_meta_modelers_guide_composites_fig04a_interaction_modalities` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig04a_interaction_modalities = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig04a-interaction-modalities.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig04a_interaction_modalities)

In [ ]:
# === Edit parameters for composite 'fig04a-interaction-modalities' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'nutrient_exchange'  (local:NutrientExchange)
spec_viva_meta_modelers_guide_composites_fig04a_interaction_modalities['state']['nutrient_exchange']['config']['interval'] = 1.0

# process 'motile_force'  (local:MotileForce)
spec_viva_meta_modelers_guide_composites_fig04a_interaction_modalities['state']['motile_force']['config']['interval'] = 1.0

# process 'growth'  (local:Growth)
spec_viva_meta_modelers_guide_composites_fig04a_interaction_modalities['state']['growth']['config']['interval'] = 1.0

# process 'electrical_signaling'  (local:ElectricalSignaling)
spec_viva_meta_modelers_guide_composites_fig04a_interaction_modalities['state']['electrical_signaling']['config']['interval'] = 1.0

**Composite `viva_meta_modelers_guide.composites.fig04b-cellular-interface`** — `spec_viva_meta_modelers_guide_composites_fig04b_cellular_interface` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig04b_cellular_interface = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig04b-cellular-interface.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig04b_cellular_interface)

In [ ]:
# === Edit parameters for composite 'fig04b-cellular-interface' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'cell'  (local:CellularInterface)
spec_viva_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: typed-interface ===
STUDY = 'typed-interface'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**fig04a-interaction-modalities**


In [ ]:
# fig04a-interaction-modalities
show_viz(_render_one('image:visualizations/fig04a-interaction-modalities.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig04b-cellular-interface**


In [ ]:
# fig04b-cellular-interface
show_viz(_render_one('image:visualizations/fig04b-cellular-interface.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig04-illustration**


In [ ]:
# fig04-illustration
show_viz(_render_one('image:visualizations/fig04-illustration.png', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig04b-executable**


In [ ]:
# fig04b-executable
show_viz(_render_one('image:visualizations/fig04b-executable.svg', {'chart': 'image', 'caption': 'EXECUTABLE dynamics — the compiled figure, running.'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| cell-grows | kind=observable expr=max(interface.shape) over the run | op threshold condition ≥ 3.0 (dimensionless shape factor) |
| goal-directed | kind=observable expr=last(interface.objective) | op threshold condition > 1.0 |
| chemical-uptake | kind=observable expr=min(interface.chemical) | op threshold condition < 0 (net uptake) |


## Study: Closing the Loop (`closing-the-loop`)

**Question.** Does the cell–environment coupling of Fig 5 close into a genuine sense/act loop when the environment is a real spatial field — i.e. does the cell read a diffusing chemical field, act back on it through an uptake flux, and grow from what it takes up, all over one shared field store?

**Objective.** Run the Fig 5 executable with a real diffusing spatial field and measure field spread, the cell's uptake flux, and cell mass to confirm the loop closes.

**Hypothesis.** Compiling Fig 5 with an environment handler that diffuses a map[float] chemical field and a cell handler that senses-and-acts produces a closed loop: an initial point source spreads across the field, the cell's uptake flux becomes positive, and its mass increases — sensing and acting being the same coupling read in two directions.

**Claim.** Sensing and acting are two sides of one coupling: over a real diffusing map[float] field the cell draws down a local source (1.0→0.20), acts back with uptake flux 0.34, and grows (mass 1.0→1.17).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `cell-environment` | `viva_meta_modelers_guide.composites.fig05-cell-environment` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_meta_modelers_guide.composites.fig05-cell-environment`** — `spec_viva_meta_modelers_guide_composites_fig05_cell_environment` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig05_cell_environment = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig05-cell-environment.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig05_cell_environment)

In [ ]:
# === Edit parameters for composite 'fig05-cell-environment' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'reaction_diffusion'  (local:ReactionDiffusion)
spec_viva_meta_modelers_guide_composites_fig05_cell_environment['state']['reaction_diffusion']['config']['interval'] = 1.0

# process 'production_degradation'  (local:ProductionDegradation)
spec_viva_meta_modelers_guide_composites_fig05_cell_environment['state']['production_degradation']['config']['interval'] = 1.0

# process 'mechanical_stress'  (local:MechanicalStress)
spec_viva_meta_modelers_guide_composites_fig05_cell_environment['state']['mechanical_stress']['config']['interval'] = 1.0

# process 'single_cell_processes'  (local:SingleCellProcesses)
spec_viva_meta_modelers_guide_composites_fig05_cell_environment['state']['single_cell_processes']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: closing-the-loop ===
STUDY = 'closing-the-loop'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**fig05-cell-environment**


In [ ]:
# fig05-cell-environment
show_viz(_render_one('image:visualizations/fig05-cell-environment.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig05-illustration**


In [ ]:
# fig05-illustration
show_viz(_render_one('image:visualizations/fig05-illustration.png', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig05-executable**


In [ ]:
# fig05-executable
show_viz(_render_one('image:visualizations/fig05-executable.svg', {'chart': 'image', 'caption': 'EXECUTABLE dynamics — the compiled figure, running.'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| field-diffuses | kind=observable expr=last(environment.chemical_field[4]) and last(chemical_field[0]) | op threshold condition source < 0.5 AND a neighbour > 0 |
| cell-acts | kind=observable expr=last(environment.uptake_flux) | op threshold condition > 0 |
| cell-grows | kind=observable expr=last(single_cell.mass) | op threshold condition > 1.0 |


## Study: One Interface, Three Mechanisms (`one-interface-three-mechanisms`)

**Question.** Can a single metabolism interface — the ports nutrients ⇒ {biomass, energy, entropy, secretions} of Fig 6 — be realized by three genuinely different mechanisms (a lumped-yield process, a saturating-kinetic process, and a real flux-balance optimization via COBRApy) while every other part of the composite, and the interface itself, stays byte-for-byte the same?

**Objective.** Compile the Fig 6 disintegration/metabolism draft under three handler environments (coarse, kinetic, fba), run all three, and compare their biomass trajectories over an identical interface.

**Hypothesis.** Installing three different handlers on the one Fig 6 metabolism signature yields three executables that all emit the same port set (biomass/energy/entropy/secretions) but with mechanism-specific trajectories — coarse highest, kinetic lowest, FBA in between — demonstrating that mechanism is swappable behind a preserved interface (compiler law 4, handler independence).

**Claim.** One metabolism interface (nutrients ⇒ biomass, energy, entropy, secretions) is realized by THREE independent handlers — coarse (biomass 4.0), saturating-kinetic (2.67), and real COBRApy FBA (3.2) — with no change to the rest of the composite (handler independence, law 4).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `disintegration` | `viva_meta_modelers_guide.composites.fig06-disintegration` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_meta_modelers_guide.composites.fig06-disintegration`** — `spec_viva_meta_modelers_guide_composites_fig06_disintegration` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig06_disintegration = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig06-disintegration.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig06_disintegration)

In [ ]:
# === Edit parameters for composite 'fig06-disintegration' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'coarse_grained_metabolism'  (local:CoarseGrainedMetabolism)
spec_viva_meta_modelers_guide_composites_fig06_disintegration['state']['coarse_grained_metabolism']['config']['interval'] = 1.0

# process 'catalyzed_reaction_network'  (local:CatalyzedReactionNetwork)
spec_viva_meta_modelers_guide_composites_fig06_disintegration['state']['catalyzed_reaction_network']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: one-interface-three-mechanisms ===
STUDY = 'one-interface-three-mechanisms'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**fig06-disintegration**


In [ ]:
# fig06-disintegration
show_viz(_render_one('image:visualizations/fig06-disintegration.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig06-illustration**


In [ ]:
# fig06-illustration
show_viz(_render_one('image:visualizations/fig06-illustration.png', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig06-executable-coarse**


In [ ]:
# fig06-executable-coarse
show_viz(_render_one('image:visualizations/fig06-executable-coarse.svg', {'chart': 'image', 'caption': 'EXECUTABLE dynamics — the compiled figure, running.'}, RUNS_DB, STUDY_YAML))

**fig06-executable-kinetic**


In [ ]:
# fig06-executable-kinetic
show_viz(_render_one('image:visualizations/fig06-executable-kinetic.svg', {'chart': 'image', 'caption': 'EXECUTABLE dynamics — the compiled figure, running.'}, RUNS_DB, STUDY_YAML))

**fig06-executable-fba**


In [ ]:
# fig06-executable-fba
show_viz(_render_one('image:visualizations/fig06-executable-fba.svg', {'chart': 'image', 'caption': 'EXECUTABLE dynamics — the compiled figure, running.'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| all-produce-biomass | kind=observable expr=last(coarse.biomass) for coarse, kinetic, fba | op threshold condition all > 0 (4.0 / 2.67 / 3.2) |
| mechanisms-differ | kind=observable expr=max-min of {coarse, kinetic, fba} last biomass | op threshold condition > 1.0 |
| interface-preserved | kind=observable expr=port-set equality across the three compiled composites | op threshold condition identical |


## Study: Molecular Channels (`molecular-channels`)

**Question.** When a description drops to the molecular grain (Fig 7), can a single molecular mechanism act as a transducer across four typed physical channels at once — chemical, electrical, mechanical, thermal — each carried on its own port in its own unit, without the channels collapsing into one lumped output?

**Objective.** Run the Fig 7 executable and measure the four channel outputs to confirm they are simultaneously active and independently typed.

**Hypothesis.** The Fig 7 molecular mechanism, compiled under a transducer handler, drives four distinct output ports with four distinct magnitudes, confirming that the typed-channel interface survives at the molecular grain.

**Claim.** At the molecular grain, one mechanism transduces four independently-typed channels at once — chemical (0.6), electrical (0.3), mechanical (0.4) and thermal (0.7) — each an emitting port in its own unit.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `molecular-mechanism` | `viva_meta_modelers_guide.composites.fig07-molecular-mechanism` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_meta_modelers_guide.composites.fig07-molecular-mechanism`** — `spec_viva_meta_modelers_guide_composites_fig07_molecular_mechanism` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig07_molecular_mechanism = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig07-molecular-mechanism.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig07_molecular_mechanism)

In [ ]:
# === Edit parameters for composite 'fig07-molecular-mechanism' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'molecular_mechanism'  (local:MolecularMechanism)
spec_viva_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: molecular-channels ===
STUDY = 'molecular-channels'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**fig07-molecular-mechanism**


In [ ]:
# fig07-molecular-mechanism
show_viz(_render_one('image:visualizations/fig07-molecular-mechanism.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig07-illustration**


In [ ]:
# fig07-illustration
show_viz(_render_one('image:visualizations/fig07-illustration.png', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig07-executable**


In [ ]:
# fig07-executable
show_viz(_render_one('image:visualizations/fig07-executable.svg', {'chart': 'image', 'caption': 'EXECUTABLE dynamics — the compiled figure, running.'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| all-channels-active | kind=observable expr=last of chemical_out, electrical_out, mechanical_out, thermal_out | op threshold condition all > 0 |
| channels-distinct | kind=observable expr=distinct port magnitudes {0.6, 0.3, 0.4, 0.7} | op threshold condition four distinct values |


## Study: The Nested Cell (`the-nested-cell`)

**Question.** How do molecules compose into a cell? Can the Fig 8 molecular composition be authored as a deeply nested place graph — membrane, cytoplasm, nucleus, chromosome, chromatin, nucleosome — with a gene-expression cascade wired to its deepest leaves, and does compilation preserve the interface even six levels down?

**Objective.** Run the Fig 8 executable and measure the expression cascade at several depths, confirming ordered DNA→RNA→protein flow and a live deepest leaf.

**Hypothesis.** Compiling the Fig 8 nested-hierarchy draft wires an expression cascade (DNA→RNA→protein) to a six-level-deep place graph and runs it, with the deepest leaf (nucleosome.DNA) changing over time and every port preserved — the strongest test of interface preservation under nesting (law 2).

**Claim.** A six-level nested place graph (ECM→membrane→cytoplasm→nucleus→chromosome→nucleosome) carries a coupled expression cascade — DNA (1.0→1.4) → RNA (→1.41) → protein (→0.52) — with the interface preserved at the deepest leaf (law 2).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `nested-hierarchy` | `viva_meta_modelers_guide.composites.fig08-nested-hierarchy` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_meta_modelers_guide.composites.fig08-nested-hierarchy`** — `spec_viva_meta_modelers_guide_composites_fig08_nested_hierarchy` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig08_nested_hierarchy = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig08-nested-hierarchy.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig08_nested_hierarchy)

In [ ]:
# === Edit parameters for composite 'fig08-nested-hierarchy' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'transmembrane_transport'  (local:TransmembraneTransport)
spec_viva_meta_modelers_guide_composites_fig08_nested_hierarchy['state']['transmembrane_transport']['config']['interval'] = 1.0

# process 'replication_and_repair'  (local:ReplicationAndRepair)
spec_viva_meta_modelers_guide_composites_fig08_nested_hierarchy['state']['replication_and_repair']['config']['interval'] = 1.0

# process 'cell_metabolism'  (local:CellMetabolism)
spec_viva_meta_modelers_guide_composites_fig08_nested_hierarchy['state']['cell_metabolism']['config']['interval'] = 1.0

# process 'transcription'  (local:Transcription)
spec_viva_meta_modelers_guide_composites_fig08_nested_hierarchy['state']['transcription']['config']['interval'] = 1.0

# process 'translation'  (local:Translation)
spec_viva_meta_modelers_guide_composites_fig08_nested_hierarchy['state']['translation']['config']['interval'] = 1.0

# process 'subunit_assembly'  (local:SubunitAssembly)
spec_viva_meta_modelers_guide_composites_fig08_nested_hierarchy['state']['subunit_assembly']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: the-nested-cell ===
STUDY = 'the-nested-cell'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**fig08-nested-hierarchy**


In [ ]:
# fig08-nested-hierarchy
show_viz(_render_one('image:visualizations/fig08-nested-hierarchy.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig08-illustration**


In [ ]:
# fig08-illustration
show_viz(_render_one('image:visualizations/fig08-illustration.png', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig08-executable**


In [ ]:
# fig08-executable
show_viz(_render_one('image:visualizations/fig08-executable.svg', {'chart': 'image', 'caption': 'EXECUTABLE dynamics — the compiled figure, running.'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| cascade-flows | kind=observable expr=last(cytoplasm.rna) and last(cytoplasm.proteins) | op threshold condition both > 0 (1.41 / 0.52) |
| deepest-leaf-live | kind=observable expr=last(...nucleosome.DNA) vs first | op threshold condition > first (1.4 > 1.0) |
| transport-active | kind=observable expr=last(membrane.transport_flux) | op threshold condition > 0 |


## Study: Self-Made (`self-made`)

**Question.** How does a cell hold itself together? Does the Fig 9 composition express autopoiesis — metabolism, containment, and replication mutually producing one another — and does that same closure appear when each function is realized at a coarse, a self-organized, or a molecular grain?

**Objective.** Run the Fig 9a (three-grain closure) and Fig 9b (minimal cell) executables and confirm metabolism, containment, and replication are all productive at more than one grain.

**Hypothesis.** The Fig 9 executables show all three closure functions simultaneously active and expressible at multiple grains: a coarse metabolism/containment/replication triad and a molecular minimal cell (Fig 9b) that grows its own membrane and proteins — the mutual closure Maturana & Varela called autopoiesis.

**Claim.** Metabolism, containment, and replication close on one another (autopoiesis) and the same three functions appear at coarse, self-organized, and molecular grains — metabolites 4.8, membrane 1.6, replication copies 1.6/1.2.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `coarse-graining` | `viva_meta_modelers_guide.composites.fig09a-coarse-graining` | 0 | — |
| `minimal-cell` | `viva_meta_modelers_guide.composites.fig09b-minimal-cell` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_meta_modelers_guide.composites.fig09a-coarse-graining`** — `spec_viva_meta_modelers_guide_composites_fig09a_coarse_graining` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig09a_coarse_graining = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig09a-coarse-graining.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig09a_coarse_graining)

In [ ]:
# === Edit parameters for composite 'fig09a-coarse-graining' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'metabolism_closure'  (local:MetabolismClosure)
spec_viva_meta_modelers_guide_composites_fig09a_coarse_graining['state']['metabolism_closure']['config']['interval'] = 1.0

# process 'autocatalysis'  (local:Autocatalysis)
spec_viva_meta_modelers_guide_composites_fig09a_coarse_graining['state']['autocatalysis']['config']['interval'] = 1.0

# process 'containment_closure'  (local:ContainmentClosure)
spec_viva_meta_modelers_guide_composites_fig09a_coarse_graining['state']['containment_closure']['config']['interval'] = 1.0

# process 'membrane_self_assembly'  (local:MembraneSelfAssembly)
spec_viva_meta_modelers_guide_composites_fig09a_coarse_graining['state']['membrane_self_assembly']['config']['interval'] = 1.0

# process 'lipid_aggregation'  (local:LipidAggregation)
spec_viva_meta_modelers_guide_composites_fig09a_coarse_graining['state']['lipid_aggregation']['config']['interval'] = 1.0

# process 'replication_closure'  (local:ReplicationClosure)
spec_viva_meta_modelers_guide_composites_fig09a_coarse_graining['state']['replication_closure']['config']['interval'] = 1.0

# process 'template_replication'  (local:TemplateReplication)
spec_viva_meta_modelers_guide_composites_fig09a_coarse_graining['state']['template_replication']['config']['interval'] = 1.0

# process 'template_directed_synthesis'  (local:TemplateDirectedSynthesis)
spec_viva_meta_modelers_guide_composites_fig09a_coarse_graining['state']['template_directed_synthesis']['config']['interval'] = 1.0

**Composite `viva_meta_modelers_guide.composites.fig09b-minimal-cell`** — `spec_viva_meta_modelers_guide_composites_fig09b_minimal_cell` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig09b_minimal_cell = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig09b-minimal-cell.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig09b_minimal_cell)

In [ ]:
# === Edit parameters for composite 'fig09b-minimal-cell' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'minimal_cell_containment'  (local:MinimalCellContainment)
spec_viva_meta_modelers_guide_composites_fig09b_minimal_cell['state']['minimal_cell_containment']['config']['interval'] = 1.0

# process 'minimal_cell_metabolism'  (local:MinimalCellMetabolism)
spec_viva_meta_modelers_guide_composites_fig09b_minimal_cell['state']['minimal_cell_metabolism']['config']['interval'] = 1.0

# process 'gene_expression'  (local:GeneExpression)
spec_viva_meta_modelers_guide_composites_fig09b_minimal_cell['state']['gene_expression']['config']['interval'] = 1.0

# process 'minimal_cell_replication'  (local:MinimalCellReplication)
spec_viva_meta_modelers_guide_composites_fig09b_minimal_cell['state']['minimal_cell_replication']['config']['interval'] = 1.0

# process 'diffusion'  (local:Diffusion)
spec_viva_meta_modelers_guide_composites_fig09b_minimal_cell['state']['diffusion']['config']['interval'] = 1.0

# process 'reactions'  (local:Reactions)
spec_viva_meta_modelers_guide_composites_fig09b_minimal_cell['state']['reactions']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: self-made ===
STUDY = 'self-made'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**fig09a-coarse-graining**


In [ ]:
# fig09a-coarse-graining
show_viz(_render_one('image:visualizations/fig09a-coarse-graining.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig09b-minimal-cell**


In [ ]:
# fig09b-minimal-cell
show_viz(_render_one('image:visualizations/fig09b-minimal-cell.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig09-illustration**


In [ ]:
# fig09-illustration
show_viz(_render_one('image:visualizations/fig09-illustration.png', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig09-illustration-2**


In [ ]:
# fig09-illustration-2
show_viz(_render_one('image:visualizations/fig09-illustration-2.png', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig09a-executable**


In [ ]:
# fig09a-executable
show_viz(_render_one('image:visualizations/fig09a-executable.svg', {'chart': 'image', 'caption': 'EXECUTABLE dynamics — the compiled figure, running.'}, RUNS_DB, STUDY_YAML))

**fig09b-executable**


In [ ]:
# fig09b-executable
show_viz(_render_one('image:visualizations/fig09b-executable.svg', {'chart': 'image', 'caption': 'EXECUTABLE dynamics — the compiled figure, running.'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| closure-active | kind=observable expr=last(metabolism_selforg.products), containment_selforg.membrane, replication_coarse.copies | op threshold condition all > 0 (2.4 / 1.6 / 1.6) |
| multi-grain | kind=observable expr=coarse and selforg variants of each function | op threshold condition both > 0 |
| minimal-cell-grows | kind=observable expr=last(membrane.area) and last(proteins.concentration) | op threshold condition both > 1.0 (1.2 / 1.27) |


## Study: Divide (`divide`)

**Question.** Is cell division in Fig 10 a genuine structural rewrite of the place graph — a single cell node actually becoming two daughter nodes at runtime — rather than a pre-declared post-structure that is merely animated?

**Objective.** Run the Fig 10-1 division executable and confirm a genuine node-creating rewrite: cell count reaching 2, symmetric DNA partition, daughters born mid-run.

**Hypothesis.** Compiling Fig 10's division draft with a Milner-style reaction rule fires a discrete event when the cell is large enough: cell_count steps from 1 to 2, two daughter nodes are created that did not exist at t=0, and the parent's DNA is partitioned symmetrically between them.

**Claim.** Division is a genuine event-driven graph rewrite: one cell node becomes two (cell_count 1→2), partitioning DNA symmetrically (2.745 each) into daughters that did not exist at t=0.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `division` | `viva_meta_modelers_guide.composites.fig10-1-division` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_meta_modelers_guide.composites.fig10-1-division`** — `spec_viva_meta_modelers_guide_composites_fig10_1_division` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig10_1_division = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig10-1-division.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig10_1_division)

In [ ]:
# === Edit parameters for composite 'fig10-1-division' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'dna_replication'  (local:DNAReplication)
spec_viva_meta_modelers_guide_composites_fig10_1_division['state']['dna_replication']['config']['interval'] = 1.0

# process 'segregate_chromosome'  (local:SegregateChromosome)
spec_viva_meta_modelers_guide_composites_fig10_1_division['state']['segregate_chromosome']['config']['interval'] = 1.0

# process 'divide'  (local:Divide)
spec_viva_meta_modelers_guide_composites_fig10_1_division['state']['divide']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: divide ===
STUDY = 'divide'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**fig10-1-division**


In [ ]:
# fig10-1-division
show_viz(_render_one('image:visualizations/fig10-1-division.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig10-illustration**


In [ ]:
# fig10-illustration
show_viz(_render_one('image:visualizations/fig10-illustration.png', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig10-1-executable**


In [ ]:
# fig10-1-executable
show_viz(_render_one('image:visualizations/fig10-1-executable.svg', {'chart': 'image', 'caption': 'EXECUTABLE dynamics — the compiled figure, running.'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| division-fires | kind=observable expr=max(environ.cell_count) | op threshold condition ≥ 2 |
| symmetric-partition | kind=observable expr=last(daughter_1.dna) vs last(daughter_2.dna) | op threshold condition equal (2.745 = 2.745) |
| nodes-created | kind=observable expr=first(daughter_1.dna) | op threshold condition = 0 at t0, > 0 after the event |


## Study: Biofilm (`biofilm`)

**Question.** Can multicellular development (Fig 10) be expressed as compositional reorganization — individual cells attaching, secreting extracellular matrix, and assembling into a biofilm that is itself a higher-level composite with its own aggregate observables?

**Objective.** Run the Fig 10-2 development executable and measure attachment, ECM secretion, and aggregate biofilm mass.

**Hypothesis.** The Fig 10-2 development executable grows a biofilm as a higher-level composite: attached-cell count, adhesion, and ECM all rise, and an aggregate biofilm_mass accumulates — development as composition, not a single process.

**Claim.** Development is composition at a higher level: cells attach (1.35), secrete ECM (1.8), and accumulate into a biofilm whose mass grows to 2.25.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `development` | `viva_meta_modelers_guide.composites.fig10-2-development` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_meta_modelers_guide.composites.fig10-2-development`** — `spec_viva_meta_modelers_guide_composites_fig10_2_development` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig10_2_development = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig10-2-development.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig10_2_development)

In [ ]:
# === Edit parameters for composite 'fig10-2-development' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'surface_attachment'  (local:SurfaceAttachment)
spec_viva_meta_modelers_guide_composites_fig10_2_development['state']['surface_attachment']['config']['interval'] = 1.0

# process 'ecm_secretion'  (local:ECMSecretion)
spec_viva_meta_modelers_guide_composites_fig10_2_development['state']['ecm_secretion']['config']['interval'] = 1.0

# process 'biofilm_growth'  (local:BiofilmGrowth)
spec_viva_meta_modelers_guide_composites_fig10_2_development['state']['biofilm_growth']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: biofilm ===
STUDY = 'biofilm'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**fig10-2-development**


In [ ]:
# fig10-2-development
show_viz(_render_one('image:visualizations/fig10-2-development.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig10-illustration-2**


In [ ]:
# fig10-illustration-2
show_viz(_render_one('image:visualizations/fig10-illustration-2.png', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig10-2-executable**


In [ ]:
# fig10-2-executable
show_viz(_render_one('image:visualizations/fig10-2-executable.svg', {'chart': 'image', 'caption': 'EXECUTABLE dynamics — the compiled figure, running.'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| biofilm-accumulates | kind=observable expr=last(environ.biofilm.biofilm_mass) | op threshold condition > 1.0 (2.25) |
| cells-attach-secrete | kind=observable expr=last(biofilm.attached) and last(biofilm.ecm) | op threshold condition both > 0 (1.35 / 1.8) |


## Study: Evolve (`evolve`)

**Question.** Can evolution (Fig 10) be modelled compositionally — variation and selection acting on a population, and, crucially, the *addition of a new interface port* to a lineage — so that the interface set itself changes over evolutionary time?

**Objective.** Run the Fig 10-3 evolution executable and measure differential growth (selection) and the emergence of a new interface port.

**Hypothesis.** The Fig 10-3 evolution executable selects a fitter variant (its cell count grows fastest) and introduces a new interface capability: a port that is absent at t=0 emerges with nonzero value, showing that composition can add ports, not just change their values.

**Claim.** Evolution is a compositional rewrite too: a fitter variant is selected (cell_count → 3.4) and a lineage gains an entirely new interface port (new_port emerges to 0.57).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `evolution` | `viva_meta_modelers_guide.composites.fig10-3-evolution` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_meta_modelers_guide.composites.fig10-3-evolution`** — `spec_viva_meta_modelers_guide_composites_fig10_3_evolution` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig10_3_evolution = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig10-3-evolution.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig10_3_evolution)

In [ ]:
# === Edit parameters for composite 'fig10-3-evolution' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'variation'  (local:Variation)
spec_viva_meta_modelers_guide_composites_fig10_3_evolution['state']['variation']['config']['interval'] = 1.0

# process 'selection'  (local:Selection)
spec_viva_meta_modelers_guide_composites_fig10_3_evolution['state']['selection']['config']['interval'] = 1.0

# process 'port_addition'  (local:PortAddition)
spec_viva_meta_modelers_guide_composites_fig10_3_evolution['state']['port_addition']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: evolve ===
STUDY = 'evolve'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**fig10-3-evolution**


In [ ]:
# fig10-3-evolution
show_viz(_render_one('image:visualizations/fig10-3-evolution.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig10-illustration-3**


In [ ]:
# fig10-illustration-3
show_viz(_render_one('image:visualizations/fig10-illustration-3.png', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig10-3-executable**


In [ ]:
# fig10-3-executable
show_viz(_render_one('image:visualizations/fig10-3-executable.svg', {'chart': 'image', 'caption': 'EXECUTABLE dynamics — the compiled figure, running.'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| selection | kind=observable expr=last(environ.cell_ecoli.cell_count) | op threshold condition > 1 (3.4) |
| new-port-emerges | kind=observable expr=first vs last of cell_O157.new_port | op threshold condition 0 -> > 0 (0.57) |


## Study: The Living Atlas (`the-living-atlas`)

**Question.** Do all of the paper's semantic figures actually compile to executables that run, and — the real test of composition — can the independently-authored figure mechanisms be assembled into a single whole cell that lives the paper's full arc: grow, divide, and die?

**Objective.** Render a dynamics figure for every executable (the gallery) and run the composed whole cell for 20 time units, measuring peak biomass, division time, minimum viability, and final debris.

**Hypothesis.** All 12 executable composites build and produce non-trivial dynamics, and a whole-cell composite assembled from the figure mechanisms (uptake+growth, metabolism, viability, division, disintegration) runs the full arc in one trajectory: biomass rises and peaks, cell_count reaches 2, then a thermal shock drives viability toward 0 and debris accumulates.

**Claim.** Every draft in the atlas compiles to a running executable (12/12 with dynamics), and the figures compose into ONE whole cell that grows (biomass→5.1), divides (cell_count→2 at t≈3.4), then loses viability under thermal shock (→0.02) and disintegrates into molecular debris (→4.87).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `fig06-exec-coarse` | `viva_meta_modelers_guide.composites.fig06-executable-coarse` | 0 | — |
| `fig06-exec-kinetic` | `viva_meta_modelers_guide.composites.fig06-executable-kinetic` | 0 | — |
| `fig04b-exec` | `viva_meta_modelers_guide.composites.fig04b-executable` | 0 | — |
| `fig05-exec-spatial` | `viva_meta_modelers_guide.composites.fig05-executable` | 0 | — |
| `fig06-exec-fba` | `viva_meta_modelers_guide.composites.fig06-executable-fba` | 0 | — |
| `fig07-exec-molecular` | `viva_meta_modelers_guide.composites.fig07-executable` | 0 | — |
| `fig08-exec-hierarchy` | `viva_meta_modelers_guide.composites.fig08-executable` | 0 | — |
| `fig09a-exec-coarse-graining` | `viva_meta_modelers_guide.composites.fig09a-executable` | 0 | — |
| `fig09b-exec-minimal-cell` | `viva_meta_modelers_guide.composites.fig09b-executable` | 0 | — |
| `fig10-1-exec-division` | `viva_meta_modelers_guide.composites.fig10-1-executable` | 0 | — |
| `fig10-2-exec-development` | `viva_meta_modelers_guide.composites.fig10-2-executable` | 0 | — |
| `fig10-3-exec-evolution` | `viva_meta_modelers_guide.composites.fig10-3-executable` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_meta_modelers_guide.composites.fig06-executable-coarse`** — `spec_viva_meta_modelers_guide_composites_fig06_executable_coarse` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig06_executable_coarse = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig06-executable-coarse.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig06_executable_coarse)

In [ ]:
# === Edit parameters for composite 'fig06-executable-coarse' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'coarse_grained_metabolism'  (local:CoarseMetabolism)
spec_viva_meta_modelers_guide_composites_fig06_executable_coarse['state']['coarse_grained_metabolism']['config']['biomass_yield'] = 0.5
spec_viva_meta_modelers_guide_composites_fig06_executable_coarse['state']['coarse_grained_metabolism']['config']['energy_yield'] = 0.3
spec_viva_meta_modelers_guide_composites_fig06_executable_coarse['state']['coarse_grained_metabolism']['config']['entropy_rate'] = 0.1
spec_viva_meta_modelers_guide_composites_fig06_executable_coarse['state']['coarse_grained_metabolism']['config']['secretion_frac'] = 0.2
spec_viva_meta_modelers_guide_composites_fig06_executable_coarse['state']['coarse_grained_metabolism']['config']['interval'] = 1.0

# process 'catalyzed_reaction_network'  (local:KineticReactionNetwork)
spec_viva_meta_modelers_guide_composites_fig06_executable_coarse['state']['catalyzed_reaction_network']['config']['k'] = 0.2
spec_viva_meta_modelers_guide_composites_fig06_executable_coarse['state']['catalyzed_reaction_network']['config']['interval'] = 1.0

**Composite `viva_meta_modelers_guide.composites.fig06-executable-kinetic`** — `spec_viva_meta_modelers_guide_composites_fig06_executable_kinetic` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig06_executable_kinetic = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig06-executable-kinetic.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig06_executable_kinetic)

In [ ]:
# === Edit parameters for composite 'fig06-executable-kinetic' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'coarse_grained_metabolism'  (local:KineticMetabolism)
spec_viva_meta_modelers_guide_composites_fig06_executable_kinetic['state']['coarse_grained_metabolism']['config']['vmax'] = 1.0
spec_viva_meta_modelers_guide_composites_fig06_executable_kinetic['state']['coarse_grained_metabolism']['config']['km'] = 0.5
spec_viva_meta_modelers_guide_composites_fig06_executable_kinetic['state']['coarse_grained_metabolism']['config']['biomass_yield'] = 0.5
spec_viva_meta_modelers_guide_composites_fig06_executable_kinetic['state']['coarse_grained_metabolism']['config']['energy_yield'] = 0.3
spec_viva_meta_modelers_guide_composites_fig06_executable_kinetic['state']['coarse_grained_metabolism']['config']['entropy_rate'] = 0.1
spec_viva_meta_modelers_guide_composites_fig06_executable_kinetic['state']['coarse_grained_metabolism']['config']['secretion_frac'] = 0.2
spec_viva_meta_modelers_guide_composites_fig06_executable_kinetic['state']['coarse_grained_metabolism']['config']['interval'] = 1.0

# process 'catalyzed_reaction_network'  (local:KineticReactionNetwork)
spec_viva_meta_modelers_guide_composites_fig06_executable_kinetic['state']['catalyzed_reaction_network']['config']['k'] = 0.2
spec_viva_meta_modelers_guide_composites_fig06_executable_kinetic['state']['catalyzed_reaction_network']['config']['interval'] = 1.0

**Composite `viva_meta_modelers_guide.composites.fig04b-executable`** — `spec_viva_meta_modelers_guide_composites_fig04b_executable` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig04b_executable = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig04b-executable.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig04b_executable)

In [ ]:
# === Edit parameters for composite 'fig04b-executable' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'cell'  (local:CellularInterfaceHandler)
spec_viva_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['uptake_rate'] = 0.8
spec_viva_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['growth_max'] = 0.6
spec_viva_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['km'] = 0.5
spec_viva_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['shape_growth_coupling'] = 1.0
spec_viva_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['objective_yield'] = 0.5
spec_viva_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['temp_opt'] = 37.0
spec_viva_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['temp_tol'] = 5.0
spec_viva_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['viability_relax'] = 0.3
spec_viva_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['viability_init'] = 1.0
spec_viva_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['elasticity'] = 0.1
spec_viva_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['membrane_conductance'] = 0.05
spec_viva_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['thermal_conductance'] = 0.02
spec_viva_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['signaling_gain'] = 0.4
spec_viva_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['interval'] = 1.0

**Composite `viva_meta_modelers_guide.composites.fig05-executable`** — `spec_viva_meta_modelers_guide_composites_fig05_executable` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig05_executable = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig05-executable.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig05_executable)

In [ ]:
# === Edit parameters for composite 'fig05-executable' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'reaction_diffusion'  (local:SpatialDiffusion)
spec_viva_meta_modelers_guide_composites_fig05_executable['state']['reaction_diffusion']['config']['diffusivity'] = 0.2
spec_viva_meta_modelers_guide_composites_fig05_executable['state']['reaction_diffusion']['config']['interval'] = 1.0

# process 'production_degradation'  (local:ProductionDegradationField)
spec_viva_meta_modelers_guide_composites_fig05_executable['state']['production_degradation']['config']['source_index'] = 0
spec_viva_meta_modelers_guide_composites_fig05_executable['state']['production_degradation']['config']['source_rate'] = 0.05
spec_viva_meta_modelers_guide_composites_fig05_executable['state']['production_degradation']['config']['decay_rate'] = 0.01
spec_viva_meta_modelers_guide_composites_fig05_executable['state']['production_degradation']['config']['interval'] = 1.0

# process 'mechanical_stress'  (local:MechanicalRelax)
spec_viva_meta_modelers_guide_composites_fig05_executable['state']['mechanical_stress']['config']['relax_rate'] = 0.3
spec_viva_meta_modelers_guide_composites_fig05_executable['state']['mechanical_stress']['config']['interval'] = 1.0

# process 'single_cell_processes'  (local:SingleCellSpatial)
spec_viva_meta_modelers_guide_composites_fig05_executable['state']['single_cell_processes']['config']['cell_index'] = 4
spec_viva_meta_modelers_guide_composites_fig05_executable['state']['single_cell_processes']['config']['interval'] = 1.0

**Composite `viva_meta_modelers_guide.composites.fig06-executable-fba`** — `spec_viva_meta_modelers_guide_composites_fig06_executable_fba` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig06_executable_fba = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig06-executable-fba.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig06_executable_fba)

In [ ]:
# === Edit parameters for composite 'fig06-executable-fba' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'coarse_grained_metabolism'  (local:FBAMetabolism)
spec_viva_meta_modelers_guide_composites_fig06_executable_fba['state']['coarse_grained_metabolism']['config']['uptake_scale'] = 1.0
spec_viva_meta_modelers_guide_composites_fig06_executable_fba['state']['coarse_grained_metabolism']['config']['biomass_capacity'] = 0.8
spec_viva_meta_modelers_guide_composites_fig06_executable_fba['state']['coarse_grained_metabolism']['config']['biomass_yield'] = 0.5
spec_viva_meta_modelers_guide_composites_fig06_executable_fba['state']['coarse_grained_metabolism']['config']['energy_yield'] = 0.3
spec_viva_meta_modelers_guide_composites_fig06_executable_fba['state']['coarse_grained_metabolism']['config']['entropy_rate'] = 0.1
spec_viva_meta_modelers_guide_composites_fig06_executable_fba['state']['coarse_grained_metabolism']['config']['secretion_frac'] = 0.2
spec_viva_meta_modelers_guide_composites_fig06_executable_fba['state']['coarse_grained_metabolism']['config']['interval'] = 1.0

# process 'catalyzed_reaction_network'  (local:KineticReactionNetwork)
spec_viva_meta_modelers_guide_composites_fig06_executable_fba['state']['catalyzed_reaction_network']['config']['k'] = 0.2
spec_viva_meta_modelers_guide_composites_fig06_executable_fba['state']['catalyzed_reaction_network']['config']['interval'] = 1.0

**Composite `viva_meta_modelers_guide.composites.fig07-executable`** — `spec_viva_meta_modelers_guide_composites_fig07_executable` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig07_executable = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig07-executable.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig07_executable)

In [ ]:
# === Edit parameters for composite 'fig07-executable' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'molecular_mechanism'  (local:MolecularMechanismHandler)
spec_viva_meta_modelers_guide_composites_fig07_executable['state']['molecular_mechanism']['config']['catalysis'] = 0.6
spec_viva_meta_modelers_guide_composites_fig07_executable['state']['molecular_mechanism']['config']['dissipation'] = 0.2
spec_viva_meta_modelers_guide_composites_fig07_executable['state']['molecular_mechanism']['config']['conductance'] = 0.3
spec_viva_meta_modelers_guide_composites_fig07_executable['state']['molecular_mechanism']['config']['coupling'] = 0.4
spec_viva_meta_modelers_guide_composites_fig07_executable['state']['molecular_mechanism']['config']['interval'] = 1.0

**Composite `viva_meta_modelers_guide.composites.fig08-executable`** — `spec_viva_meta_modelers_guide_composites_fig08_executable` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig08_executable = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig08-executable.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig08_executable)

In [ ]:
# === Edit parameters for composite 'fig08-executable' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'transmembrane_transport'  (local:TransmembraneTransportODE)
spec_viva_meta_modelers_guide_composites_fig08_executable['state']['transmembrane_transport']['config']['k'] = 0.3
spec_viva_meta_modelers_guide_composites_fig08_executable['state']['transmembrane_transport']['config']['metabolite_frac'] = 0.1
spec_viva_meta_modelers_guide_composites_fig08_executable['state']['transmembrane_transport']['config']['interval'] = 1.0

# process 'replication_and_repair'  (local:ReplicationAndRepairODE)
spec_viva_meta_modelers_guide_composites_fig08_executable['state']['replication_and_repair']['config']['k'] = 0.05
spec_viva_meta_modelers_guide_composites_fig08_executable['state']['replication_and_repair']['config']['interval'] = 1.0

# process 'cell_metabolism'  (local:CellMetabolismODE)
spec_viva_meta_modelers_guide_composites_fig08_executable['state']['cell_metabolism']['config']['k'] = 0.25
spec_viva_meta_modelers_guide_composites_fig08_executable['state']['cell_metabolism']['config']['metabolite_yield'] = 0.6
spec_viva_meta_modelers_guide_composites_fig08_executable['state']['cell_metabolism']['config']['energy_yield'] = 0.4
spec_viva_meta_modelers_guide_composites_fig08_executable['state']['cell_metabolism']['config']['interval'] = 1.0

# process 'transcription'  (local:TranscriptionODE)
spec_viva_meta_modelers_guide_composites_fig08_executable['state']['transcription']['config']['k'] = 0.3
spec_viva_meta_modelers_guide_composites_fig08_executable['state']['transcription']['config']['interval'] = 1.0

# process 'translation'  (local:TranslationODE)
spec_viva_meta_modelers_guide_composites_fig08_executable['state']['translation']['config']['k'] = 0.4
spec_viva_meta_modelers_guide_composites_fig08_executable['state']['translation']['config']['km'] = 0.5
spec_viva_meta_modelers_guide_composites_fig08_executable['state']['translation']['config']['interval'] = 1.0

# process 'subunit_assembly'  (local:SubunitAssemblyODE)
spec_viva_meta_modelers_guide_composites_fig08_executable['state']['subunit_assembly']['config']['k'] = 0.2
spec_viva_meta_modelers_guide_composites_fig08_executable['state']['subunit_assembly']['config']['interval'] = 1.0

**Composite `viva_meta_modelers_guide.composites.fig09a-executable`** — `spec_viva_meta_modelers_guide_composites_fig09a_executable` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig09a_executable = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig09a-executable.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig09a_executable)

In [ ]:
# === Edit parameters for composite 'fig09a-executable' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'metabolism_closure'  (local:MetabolismClosureODE)
spec_viva_meta_modelers_guide_composites_fig09a_executable['state']['metabolism_closure']['config']['metabolite_yield'] = 0.6
spec_viva_meta_modelers_guide_composites_fig09a_executable['state']['metabolism_closure']['config']['entropy_rate'] = 0.1
spec_viva_meta_modelers_guide_composites_fig09a_executable['state']['metabolism_closure']['config']['interval'] = 1.0

# process 'autocatalysis'  (local:AutocatalysisODE)
spec_viva_meta_modelers_guide_composites_fig09a_executable['state']['autocatalysis']['config']['k'] = 0.3
spec_viva_meta_modelers_guide_composites_fig09a_executable['state']['autocatalysis']['config']['k_cat'] = 0.15
spec_viva_meta_modelers_guide_composites_fig09a_executable['state']['autocatalysis']['config']['interval'] = 1.0

# process 'containment_closure'  (local:ContainmentClosureODE)
spec_viva_meta_modelers_guide_composites_fig09a_executable['state']['containment_closure']['config']['assembly_rate'] = 0.15
spec_viva_meta_modelers_guide_composites_fig09a_executable['state']['containment_closure']['config']['perm_max'] = 0.8
spec_viva_meta_modelers_guide_composites_fig09a_executable['state']['containment_closure']['config']['perm_km'] = 1.0
spec_viva_meta_modelers_guide_composites_fig09a_executable['state']['containment_closure']['config']['interval'] = 1.0

# process 'membrane_self_assembly'  (local:MembraneSelfAssemblyODE)
spec_viva_meta_modelers_guide_composites_fig09a_executable['state']['membrane_self_assembly']['config']['k'] = 0.2
spec_viva_meta_modelers_guide_composites_fig09a_executable['state']['membrane_self_assembly']['config']['interval'] = 1.0

# process 'lipid_aggregation'  (local:LipidAggregationODE)
spec_viva_meta_modelers_guide_composites_fig09a_executable['state']['lipid_aggregation']['config']['k'] = 0.1
spec_viva_meta_modelers_guide_composites_fig09a_executable['state']['lipid_aggregation']['config']['interval'] = 1.0

# process 'replication_closure'  (local:ReplicationClosureODE)
spec_viva_meta_modelers_guide_composites_fig09a_executable['state']['replication_closure']['config']['k'] = 0.2
spec_viva_meta_modelers_guide_composites_fig09a_executable['state']['replication_closure']['config']['interval'] = 1.0

# process 'template_replication'  (local:TemplateReplicationODE)
spec_viva_meta_modelers_guide_composites_fig09a_executable['state']['template_replication']['config']['k'] = 0.15
spec_viva_meta_modelers_guide_composites_fig09a_executable['state']['template_replication']['config']['interval'] = 1.0

# process 'template_directed_synthesis'  (local:TemplateDirectedSynthesisProc)
spec_viva_meta_modelers_guide_composites_fig09a_executable['state']['template_directed_synthesis']['config']['interval'] = 1.0

**Composite `viva_meta_modelers_guide.composites.fig09b-executable`** — `spec_viva_meta_modelers_guide_composites_fig09b_executable` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig09b_executable = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig09b-executable.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig09b_executable)

In [ ]:
# === Edit parameters for composite 'fig09b-executable' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'minimal_cell_containment'  (local:ContainmentODE)
spec_viva_meta_modelers_guide_composites_fig09b_executable['state']['minimal_cell_containment']['config']['assembly_rate'] = 0.15
spec_viva_meta_modelers_guide_composites_fig09b_executable['state']['minimal_cell_containment']['config']['interval'] = 1.0

# process 'minimal_cell_metabolism'  (local:MetabolismLinear)
spec_viva_meta_modelers_guide_composites_fig09b_executable['state']['minimal_cell_metabolism']['config']['k_cat'] = 0.2
spec_viva_meta_modelers_guide_composites_fig09b_executable['state']['minimal_cell_metabolism']['config']['metabolite_yield'] = 0.6
spec_viva_meta_modelers_guide_composites_fig09b_executable['state']['minimal_cell_metabolism']['config']['energy_yield'] = 0.4
spec_viva_meta_modelers_guide_composites_fig09b_executable['state']['minimal_cell_metabolism']['config']['interval'] = 1.0

# process 'gene_expression'  (local:GeneExpressionODE)
spec_viva_meta_modelers_guide_composites_fig09b_executable['state']['gene_expression']['config']['k_expr'] = 0.25
spec_viva_meta_modelers_guide_composites_fig09b_executable['state']['gene_expression']['config']['protein_yield'] = 0.5
spec_viva_meta_modelers_guide_composites_fig09b_executable['state']['gene_expression']['config']['enzyme_yield'] = 0.3
spec_viva_meta_modelers_guide_composites_fig09b_executable['state']['gene_expression']['config']['interval'] = 1.0

# process 'minimal_cell_replication'  (local:ReplicationODE)
spec_viva_meta_modelers_guide_composites_fig09b_executable['state']['minimal_cell_replication']['config']['k_rep'] = 0.1
spec_viva_meta_modelers_guide_composites_fig09b_executable['state']['minimal_cell_replication']['config']['gene_yield'] = 0.5
spec_viva_meta_modelers_guide_composites_fig09b_executable['state']['minimal_cell_replication']['config']['nucleic_yield'] = 0.4
spec_viva_meta_modelers_guide_composites_fig09b_executable['state']['minimal_cell_replication']['config']['interval'] = 1.0

# process 'diffusion'  (local:DiffusionRelax)
spec_viva_meta_modelers_guide_composites_fig09b_executable['state']['diffusion']['config']['turnover_rate'] = 0.05
spec_viva_meta_modelers_guide_composites_fig09b_executable['state']['diffusion']['config']['interval'] = 1.0

# process 'reactions'  (local:MassActionReactions)
spec_viva_meta_modelers_guide_composites_fig09b_executable['state']['reactions']['config']['k_react'] = 0.15
spec_viva_meta_modelers_guide_composites_fig09b_executable['state']['reactions']['config']['protein_yield'] = 0.4
spec_viva_meta_modelers_guide_composites_fig09b_executable['state']['reactions']['config']['nucleic_turnover'] = 0.2
spec_viva_meta_modelers_guide_composites_fig09b_executable['state']['reactions']['config']['interval'] = 1.0

**Composite `viva_meta_modelers_guide.composites.fig10-1-executable`** — `spec_viva_meta_modelers_guide_composites_fig10_1_executable` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig10_1_executable = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig10-1-executable.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig10_1_executable)

In [ ]:
# === Edit parameters for composite 'fig10-1-executable' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'dna_replication'  (local:DNAReplicationODE)
spec_viva_meta_modelers_guide_composites_fig10_1_executable['state']['dna_replication']['config']['k'] = 0.15
spec_viva_meta_modelers_guide_composites_fig10_1_executable['state']['dna_replication']['config']['interval'] = 1.0

# process 'segregate_chromosome'  (local:SegregateChromosomeProc)
spec_viva_meta_modelers_guide_composites_fig10_1_executable['state']['segregate_chromosome']['config']['seg_rate'] = 0.2
spec_viva_meta_modelers_guide_composites_fig10_1_executable['state']['segregate_chromosome']['config']['interval'] = 1.0

# process 'divide'  (local:DivisionRewrite)
spec_viva_meta_modelers_guide_composites_fig10_1_executable['state']['divide']['config']['division_time'] = 5.0
spec_viva_meta_modelers_guide_composites_fig10_1_executable['state']['divide']['config']['interval'] = 1.0

**Composite `viva_meta_modelers_guide.composites.fig10-2-executable`** — `spec_viva_meta_modelers_guide_composites_fig10_2_executable` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig10_2_executable = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig10-2-executable.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig10_2_executable)

In [ ]:
# === Edit parameters for composite 'fig10-2-executable' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'surface_attachment'  (local:SurfaceAttachmentProc)
spec_viva_meta_modelers_guide_composites_fig10_2_executable['state']['surface_attachment']['config']['attach_rate'] = 0.15
spec_viva_meta_modelers_guide_composites_fig10_2_executable['state']['surface_attachment']['config']['adhesion_coef'] = 0.5
spec_viva_meta_modelers_guide_composites_fig10_2_executable['state']['surface_attachment']['config']['interval'] = 1.0

# process 'ecm_secretion'  (local:ECMSecretionProc)
spec_viva_meta_modelers_guide_composites_fig10_2_executable['state']['ecm_secretion']['config']['k'] = 0.2
spec_viva_meta_modelers_guide_composites_fig10_2_executable['state']['ecm_secretion']['config']['interval'] = 1.0

# process 'biofilm_growth'  (local:BiofilmGrowthProc)
spec_viva_meta_modelers_guide_composites_fig10_2_executable['state']['biofilm_growth']['config']['mass_rate'] = 0.25
spec_viva_meta_modelers_guide_composites_fig10_2_executable['state']['biofilm_growth']['config']['growth_rate'] = 0.08
spec_viva_meta_modelers_guide_composites_fig10_2_executable['state']['biofilm_growth']['config']['interval'] = 1.0

**Composite `viva_meta_modelers_guide.composites.fig10-3-executable`** — `spec_viva_meta_modelers_guide_composites_fig10_3_executable` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_meta_modelers_guide_composites_fig10_3_executable = load_spec(REPO / 'viva_meta_modelers_guide/composites/fig10-3-executable.composite.json')
describe_spec(spec_viva_meta_modelers_guide_composites_fig10_3_executable)

In [ ]:
# === Edit parameters for composite 'fig10-3-executable' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'variation'  (local:VariationProc)
spec_viva_meta_modelers_guide_composites_fig10_3_executable['state']['variation']['config']['interval'] = 1.0

# process 'selection'  (local:SelectionProc)
spec_viva_meta_modelers_guide_composites_fig10_3_executable['state']['selection']['config']['k'] = 0.3
spec_viva_meta_modelers_guide_composites_fig10_3_executable['state']['selection']['config']['interval'] = 1.0

# process 'port_addition'  (local:PortAdditionProc)
spec_viva_meta_modelers_guide_composites_fig10_3_executable['state']['port_addition']['config']['onset_rate'] = 0.1
spec_viva_meta_modelers_guide_composites_fig10_3_executable['state']['port_addition']['config']['capacity'] = 1.0
spec_viva_meta_modelers_guide_composites_fig10_3_executable['state']['port_addition']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: the-living-atlas ===
STUDY = 'the-living-atlas'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**fig04b-executable**


In [ ]:
# fig04b-executable
show_viz(_render_one('image:visualizations/fig04b-executable.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig05-executable**


In [ ]:
# fig05-executable
show_viz(_render_one('image:visualizations/fig05-executable.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig06-executable-coarse**


In [ ]:
# fig06-executable-coarse
show_viz(_render_one('image:visualizations/fig06-executable-coarse.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig06-executable-kinetic**


In [ ]:
# fig06-executable-kinetic
show_viz(_render_one('image:visualizations/fig06-executable-kinetic.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig06-executable-fba**


In [ ]:
# fig06-executable-fba
show_viz(_render_one('image:visualizations/fig06-executable-fba.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig07-executable**


In [ ]:
# fig07-executable
show_viz(_render_one('image:visualizations/fig07-executable.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig08-executable**


In [ ]:
# fig08-executable
show_viz(_render_one('image:visualizations/fig08-executable.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig09a-executable**


In [ ]:
# fig09a-executable
show_viz(_render_one('image:visualizations/fig09a-executable.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig09b-executable**


In [ ]:
# fig09b-executable
show_viz(_render_one('image:visualizations/fig09b-executable.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig10-1-executable**


In [ ]:
# fig10-1-executable
show_viz(_render_one('image:visualizations/fig10-1-executable.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig10-2-executable**


In [ ]:
# fig10-2-executable
show_viz(_render_one('image:visualizations/fig10-2-executable.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig10-3-executable**


In [ ]:
# fig10-3-executable
show_viz(_render_one('image:visualizations/fig10-3-executable.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**wholecell**


In [ ]:
# wholecell
show_viz(_render_one('image:visualizations/wholecell.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| all-executables-run | kind=observable expr=count of executables with a rendered dynamics series | op threshold condition = 12 / 12 |
| whole-cell-grows | kind=observable expr=max(biomass) | op threshold condition > 3.0 (5.108) |
| whole-cell-divides | kind=observable expr=max(cell_count) | op threshold condition ≥ 2 (at t ≈ 3.4) |
| whole-cell-dies | kind=observable expr=min(viability) and last(debris) | op threshold condition viability < 0.1 AND debris > 1 (0.018 / 4.87) |
